# 08 — TDBRAIN — rTMS, EEG, ECG

Évaluation du modèle **tdbrain_multi** : cohorte réelle TDBRAIN, jeu de variables **multimodal** : 4 variables cliniques + 130 puissances de bande EEG (26 canaux × 5 bandes) + 5 métriques HRV issues de l'ECG (139 variables).

Ce carnet fait partie d'une série de quatre, qui croise **deux cohortes** (simulée, réelle) et **deux jeux de variables** (clinique, multimodal). Lire les quatre ensemble sépare deux questions qu'un modèle seul ne peut pas distinguer : *le signal neurophysiologique apporte-t-il quelque chose ?* (comparer les jeux de variables) et *la cohorte simulée reproduit-elle la réalité ?* (comparer les cohortes).

| Carnet | Cohorte | Variables |
|---|---|---|
| 05 | simulée | clinique |
| 06 | TDBRAIN | clinique |
| 07 | simulée | multimodal |
| 08 | TDBRAIN | multimodal |

> **Protocole d'évaluation** — validation croisée **patient-wise** (`GroupKFold`) : aucun patient n'apparaît à la fois en entraînement et en validation. Toutes les courbes sont tracées sur les prédictions **hors échantillon**.


In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.data.modalities import build_features
from src.models.lstm import LSTMConfig
from src.models.train import TrainConfig, cross_validate
from src.models.variants import Variant, variant_config
from src.reporting import model_charts as mc

CFG = variant_config(Variant.TDBRAIN_MULTI)
print(CFG.label, '|', '+'.join(CFG.modalities))


## 1. La cohorte

Cohorte **réelle** TDBRAIN : 132 patients dépressifs traités par rTMS (protocoles 1 et 2), un enregistrement de repos avant traitement par patient, 26 canaux EEG + dérivation ECG.

> Les données sont soumises à un accord d'utilisation et ne sont pas dans le dépôt. Si elles sont absentes, ce carnet bascule sur le jeu **synthétique de test** afin de rester exécutable — les chiffres ne sont alors pas ceux de la cohorte réelle, et un avertissement le signale.


In [ ]:
from src.data.tdbrain import (
    TDBRAINConfig, load_tdbrain, make_synthetic_tdbrain,
)

TDBRAIN_ROOT = ROOT / 'data/tdbrain/TDBRAIN_Dataset_V3_1_Encr/TDBRAIN_Dataset_V3_1'
REAL = (TDBRAIN_ROOT / 'participants.tsv').exists()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    if REAL:
        ds = load_tdbrain(TDBRAINConfig(
            root=TDBRAIN_ROOT, col_id='TDBRAIN_ID',
            col_protocol='rTMS PROTOCOL',
        ))
    else:
        import tempfile
        print('!! DONNEES REELLES ABSENTES — repli sur le jeu synthetique !!')
        root = Path(tempfile.mkdtemp()) / 'td'
        make_synthetic_tdbrain(root, n_patients=40, seed=1,
                               with_ecg=True, duration_seconds=20.0)
        ds = load_tdbrain(TDBRAINConfig(
            root=root, n_epochs=4, epoch_seconds=1.0, target_fs=250.0,
        ))

print('cohorte reelle' if REAL else 'JEU SYNTHETIQUE (demonstration)')
print(f'{ds.signals_mc.shape[0]} patients · {ds.signals_mc.shape[1]} époques '
      f'· {ds.signals_mc.shape[2]} canaux · fs {ds.fs:g} Hz')
print(f'répondeurs : {int(ds.labels.sum())}/{len(ds.labels)} '
      f'({ds.labels.mean():.1%})')


### Composition de la cohorte

Le **taux de base** — la proportion de la classe majoritaire — est la référence à battre. Un modèle qui prédit toujours « répondeur » atteint mécaniquement cette exactitude sans rien avoir appris.


In [ ]:
md = ds.metadata
base_rate = float(max(ds.labels.mean(), 1 - ds.labels.mean()))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), constrained_layout=True)
fig.patch.set_facecolor(mc.SURFACE)

# Répartition des classes
counts = [int((ds.labels == 0).sum()), int((ds.labels == 1).sum())]
axes[0].bar(['Non-répondeur', 'Répondeur'], counts,
            color=[mc.SERIES[1], mc.SERIES[0]], width=0.6)
for i, c in enumerate(counts):
    axes[0].annotate(str(c), xy=(i, c), xytext=(0, 4),
                     textcoords='offset points', ha='center',
                     fontsize=10, fontweight='bold', color=mc.INK)
mc.style_axes(axes[0], f'Classes (taux de base {base_rate:.1%})', '', 'Patients')

# Âge par classe — la variable la plus discriminante de la cohorte réelle
for lbl, name, colour in ((1, 'Répondeur', mc.SERIES[0]),
                          (0, 'Non-répondeur', mc.SERIES[1])):
    axes[1].hist(md.loc[ds.labels == lbl, 'age'], bins=12, alpha=0.65,
                 color=colour, label=name)
leg = axes[1].legend(frameon=False, fontsize=9)
for t in leg.get_texts():
    t.set_color(mc.INK_2)
mc.style_axes(axes[1], 'Âge par classe', 'Âge (années)', 'Patients')

# BDI-II de référence par classe — ne sépare pas, sur données réelles
for lbl, name, colour in ((1, 'Répondeur', mc.SERIES[0]),
                          (0, 'Non-répondeur', mc.SERIES[1])):
    axes[2].hist(md.loc[ds.labels == lbl, 'bdi_pre'], bins=12, alpha=0.65,
                 color=colour, label=name)
leg = axes[2].legend(frameon=False, fontsize=9)
for t in leg.get_texts():
    t.set_color(mc.INK_2)
mc.style_axes(axes[2], 'BDI-II de référence par classe', 'BDI-II', 'Patients')
plt.show()

print(f"âge      — répondeurs {md.loc[ds.labels==1,'age'].mean():.1f} "
      f"vs non-répondeurs {md.loc[ds.labels==0,'age'].mean():.1f}")
print(f"BDI_pre  — répondeurs {md.loc[ds.labels==1,'bdi_pre'].mean():.1f} "
      f"vs non-répondeurs {md.loc[ds.labels==0,'bdi_pre'].mean():.1f}")


## 2. Construction des variables

Les blocs sont assemblés par `src.data.modalities.build_features`, utilisée à l'identique pour les quatre modèles. L'ordre des blocs est **canonique** (`rtms, eeg, ecg`) quel que soit l'ordre demandé : un vecteur permuté serait silencieux et fatal.

Les blocs **clinique** et **HRV** sont constants d'une époque à l'autre (ce sont des propriétés du patient) : ils ne sont **jamais** normalisés par z-score intra-patient, qui les réduirait à zéro. Seul le bloc EEG l'est.


In [ ]:
x, y, groups, names = build_features(
    ds, modalities=CFG.modalities, per_patient_zscore=True,
)
print(f'x = {x.shape}  (patients, époques, variables)')
print(f'{len(names)} variables — {names[:4]} … {names[-3:]}')


## 3. Validation croisée

`GroupKFold` sur l'identifiant patient. Les probabilités hors échantillon de chaque pli sont conservées : chaque patient est ainsi noté exactement une fois, par un modèle qui ne l'a jamais vu.


In [ ]:
cv = cross_validate(
    x, y.astype(np.float32), groups,
    lstm_cfg=LSTMConfig(input_size=x.shape[-1]),
    train_cfg=TrainConfig(epochs=30),
    n_splits=5,
)
summary = cv.summary()
for k in ('auc_mean', 'auc_std', 'accuracy_mean', 'f1_mean'):
    print(f'{k:<15} {summary[k]:.4f}')
print(f'{"taux de base":<15} {base_rate:.4f}')


## 4. Rapport d'évaluation

Six panneaux. Deux points de lecture :

- **Deux AUC apparaissent, et elles diffèrent.** Le panneau de verdict montre la *moyenne des AUC par pli* ; la courbe ROC montre l'*AUC groupée* sur toutes les prédictions hors échantillon. Ce sont deux quantités légitimes et différentes.
- **L'exactitude se lit contre le taux de base.** Une exactitude égale au taux de base signifie que le modèle prédit toujours la même classe — la matrice de confusion le rend visible (une colonne vide).


In [ ]:
fig = mc.model_report(
    CFG.label, cv, y, x, groups, names, base_rate=base_rate,
)
plt.show()


### Courbes d'apprentissage

Perte d'entraînement et de validation pour chaque pli. Un écart qui se creuse indique un surapprentissage ; deux courbes plates indiquent que le modèle n'a rien trouvé à apprendre.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2), constrained_layout=True)
fig.patch.set_facecolor(mc.SURFACE)
mc.plot_learning_curves(ax, cv.folds)
plt.show()


## Conclusion

À compléter à la lecture des sorties ci-dessus. Les trois questions à trancher :

1. L'AUC dépasse-t-elle 0,5 **en tenant compte de la dispersion entre plis** ? Une moyenne au-dessus du hasard dont l'écart-type recouvre 0,5 n'établit rien sur cet effectif.
2. L'exactitude dépasse-t-elle le **taux de base** ? Sinon, le modèle prédit toujours la classe majoritaire.
3. Le résultat est-il **cohérent avec le carnet apparié** (même jeu de variables, autre cohorte) ?

> Un résultat négatif obtenu sous un protocole propre est un résultat publiable. Il ne doit pas être masqué par un choix de seuil ou une métrique flatteuse.
